# 05 · Forecasting & Insight Generation
**Climate Trend Analyzer** — step 5 of 5

### Forecasting done properly
The most common mistake in portfolio projects is *fitting a model and
plotting it* without ever checking whether it can predict. Here every
candidate model is scored on the **same 60-month chronological validation window** that was
not used during training, but used to select the model:

| Model | Idea |
| --- | --- |
| Naive | "next month = this month" - the benchmark everyone must beat |
| Seasonal naive | "next month = the same month last year" |
| Drift | "the historical slope continues" |
| Ridge + calendar/time features | the machine-learning approach |
| ARIMA | the classic statistical time-series model |

The winner is then refit on the full history to project the next 36 months
with an approximate 95 % prediction band.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Launch Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

from src import config as cfg
from src import data_loader, preprocessing, features, eda, trend_analysis
from src import anomaly_detection, forecasting, insights, visualizations as viz

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)

Project root: /home/user/Climate-Trend-Analyzer


In [2]:
feat = pd.read_csv(cfg.FEATURED_CSV, parse_dates=["date"])

## 5.1 Compare the candidate models

In [3]:
res = forecasting.run_forecasting(feat, target="temp_mean_c", h=36, test_size=60)
res["metrics"].round(3)

,model,MAE,RMSE,R2,n_evaluated,region
0,Ridge (calendar + trend),1.153,1.637,0.845,59,Hyderabad
1,"ARIMA(2, 0, 2)",1.566,2.021,0.763,59,Hyderabad
2,Seasonal naive,1.705,2.224,0.714,59,Hyderabad
3,Drift (linear),3.672,4.254,-0.048,59,Hyderabad
4,Naive,3.677,4.257,-0.049,59,Hyderabad
5,Ridge (calendar + trend),1.335,1.689,0.893,58,London
6,"ARIMA(2, 0, 2)",1.924,2.444,0.777,58,London
7,Seasonal naive,2.263,2.760,0.715,58,London
8,Drift (linear),6.746,8.066,-1.433,58,London
9,Naive,6.838,8.168,-1.495,58,London


In [4]:
viz.plot_model_comparison(res["metrics"])

[viz] saved 15_model_comparison.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/15_model_comparison.png')

Note how badly the naive baselines do versus the models that understand
seasonality and autocorrelation - that comparison is the real content of
this section.

## 5.2 The forecast

In [5]:
res["forecast"].head(12).round(2)

,region,date,prediction,ci_low,ci_high,model,target,interval_method
0,Hyderabad,2025-01-01,28.69,25.49,31.90,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
1,Hyderabad,2025-02-01,31.42,28.21,34.63,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
2,Hyderabad,2025-03-01,33.40,30.19,36.61,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
3,Hyderabad,2025-04-01,34.10,30.90,37.31,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
4,Hyderabad,2025-05-01,33.34,30.14,36.55,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
5,Hyderabad,2025-06-01,31.32,28.11,34.53,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
6,Hyderabad,2025-07-01,28.58,25.37,31.79,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
7,Hyderabad,2025-08-01,25.86,22.65,29.07,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
8,Hyderabad,2025-09-01,23.88,20.68,27.09,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...
9,Hyderabad,2025-10-01,23.18,19.98,26.39,Ridge (calendar + trend),temp_mean_c,approximate 1.96 x validation RMSE; not calibr...


In [6]:
viz.plot_forecast(res, region="London")

[viz] saved 14_forecast.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/14_forecast.png')

## 5.3 Generating the written insights automatically

In [7]:
from src import pipeline
from src import report_generator, insights as ins

bundle = {
    "featured": feat,
    "trend_results": trend_analysis.run_trend_analysis(feat),
    "anomaly_results": anomaly_detection.run_anomaly_detection(feat),
    "forecast_results": res,
    "eda": eda.run_eda(feat),
    "clean_report": preprocessing.clean_dataset(data_loader.load_raw())[1],
}
text = ins.generate_insights(bundle)
print(text[:2500])

[data_loader] Loaded raw dataset: /home/user/Climate-Trend-Analyzer/data/raw/climate_raw.csv -> shape (2715, 14)
## Key Findings

*Generated automatically on 2026-10-18 09:33 from 2,700 cleaned monthly records (1980-01 to 2024-12) across 5 regions.*

### 1. Warming trend

- **Synthetic warming signal.** 5 of 5 simulated regions show a statistically significant positive temperature trend (19 of 25 region x indicator combinations are significant at p < 0.05).
- **Fastest synthetic warming:** London at **0.398 °C per decade** (OLS slope 0.0398 °C/yr, full model R² (including seasonal effects) = 0.932, p = 3.00e-06).
- **Slowest warming:** Singapore at 0.228 °C per decade.
- **Mann-Kendall (non-parametric) test** confirms the trend for 5/5 regions (all ['increasing']).
- Regional spread in warming rate is **0.169 °C/decade**, i.e. the same climate driver produces very different local impacts - which is exactly why regionalisation matters for policy.

### 2. Seasonal structure

- **Every re

Every number in that text is read from the analysis results - the insight
engine cannot make a claim that the data does not support.

## 5.4 What a business user does with this

In [8]:
summary = ins.key_numbers(bundle)
for k, v in summary.items():
    print(f"{k:32s}: {v}")

rows_analysed                   : 2700
regions                         : 5
fastest_warming_region          : London
fastest_warming_c_per_decade    : 0.3976
mean_warming_c_per_decade       : 0.2984
anomalies_detected              : 362
best_forecast_rmse              : 1.3148
data_completeness_pct           : 100.0


### ✅ Checkpoint
* Compare validation errors with every baseline; winning is not assumed
* A 36-month projection with an approximate residual band is produced per region
* Findings are generated automatically from the numbers